In [7]:
!pip install -U transformers accelerate datasets sentence-transformers rapidfuzz rouge-score


In [8]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import json
from dataclasses import dataclass
from typing import Dict, List, Optional

import pandas as pd
import torch

gc.collect()
torch.cuda.empty_cache()

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

class Args:
    original_data = "/kaggle/input/datasets/belalheshamfathy/qa-data-ssssss/qa_data_summarized_flan_t5_final.json"
    summarized_data = None

    model_name = "google/t5-v1_1-base"
    output_dir = "/kaggle/working/t5_v11_base_generator_output"

    max_input_length = 256
    max_target_length = 32

    train_batch_size = 1
    eval_batch_size = 1
    gradient_accumulation_steps = 8

    num_epochs = 3
    learning_rate = 3e-5
    weight_decay = 0.01

    test_size = 0.15
    seed = 42


@dataclass
class Sample:
    knowledge: str
    question: str
    right_answer: str
    hallucinated_answer: str = ""
    source_type: str = "original"


def load_json_or_jsonl(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        first = f.read(1)
        f.seek(0)

        if first == "[":
            data = json.load(f)
        else:
            data = [json.loads(line) for line in f if line.strip()]

    return data


def build_samples(original_path: str, summarized_path: Optional[str] = None) -> List[Sample]:
    data = []

    for row in load_json_or_jsonl(original_path):
        data.append(
            Sample(
                knowledge=row["knowledge"],
                question=row["question"],
                right_answer=row["right_answer"],
                hallucinated_answer=row.get("hallucinated_answer", ""),
                source_type="original",
            )
        )

    if summarized_path:
        for row in load_json_or_jsonl(summarized_path):
            data.append(
                Sample(
                    knowledge=row["knowledge"],
                    question=row["question"],
                    right_answer=row["right_answer"],
                    hallucinated_answer=row.get("hallucinated_answer", ""),
                    source_type="flan_t5_summary",
                )
            )

    return data


def make_prompt(knowledge: str, question: str) -> str:
    return (
        f"question: {question}\n"
        f"context: {knowledge}\n"
        "answer:"
    )


def dataframe_from_samples(samples: List[Sample]) -> pd.DataFrame:
    rows = []

    for s in samples:
        rows.append(
            {
                "knowledge": s.knowledge,
                "question": s.question,
                "right_answer": s.right_answer,
                "hallucinated_answer": s.hallucinated_answer,
                "source_type": s.source_type,
                "input_text": make_prompt(s.knowledge, s.question),
                "target_text": s.right_answer,
            }
        )

    return pd.DataFrame(rows)


def preprocess_dataset(dataset: Dataset, tokenizer, max_input_length: int, max_target_length: int):
    def _tokenize(batch):
        model_inputs = tokenizer(
            batch["input_text"],
            max_length=max_input_length,
            truncation=True,
        )

        labels = tokenizer(
            text_target=batch["target_text"],
            max_length=max_target_length,
            truncation=True,
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    return dataset.map(
        _tokenize,
        batched=True,
        remove_columns=dataset.column_names,
    )


def main_train(args):
    gc.collect()
    torch.cuda.empty_cache()

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    print("Loading data...")
    samples = build_samples(args.original_data, args.summarized_data)
    df = dataframe_from_samples(samples)

    train_df, eval_df = train_test_split(
        df,
        test_size=args.test_size,
        random_state=args.seed,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    eval_df = eval_df.reset_index(drop=True)

    # IMPORTANT: save splits for evaluation
    train_split_path = os.path.join(args.output_dir, "train_split.csv")
    eval_split_path = os.path.join(args.output_dir, "eval_split.csv")

    train_df.to_csv(train_split_path, index=False, encoding="utf-8-sig")
    eval_df.to_csv(eval_split_path, index=False, encoding="utf-8-sig")

    print(f"Train samples: {len(train_df)} | Eval samples: {len(eval_df)}")
    print(f"Train split saved to: {train_split_path}")
    print(f"Eval split saved to: {eval_split_path}")

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name)

    print("Loading model...")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        args.model_name,
        low_cpu_mem_usage=True,
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    train_ds = Dataset.from_pandas(train_df)
    eval_ds = Dataset.from_pandas(eval_df)

    train_tok = preprocess_dataset(
        train_ds,
        tokenizer,
        args.max_input_length,
        args.max_target_length,
    )

    eval_tok = preprocess_dataset(
        eval_ds,
        tokenizer,
        args.max_input_length,
        args.max_target_length,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output_dir,

        per_device_train_batch_size=args.train_batch_size,
        per_device_eval_batch_size=args.eval_batch_size,

        gradient_accumulation_steps=args.gradient_accumulation_steps,

        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        warmup_steps=100,

        num_train_epochs=args.num_epochs,

        eval_strategy="epoch",
        save_strategy="epoch",

        logging_strategy="steps",
        logging_steps=50,

        predict_with_generate=False,

        save_total_limit=1,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        fp16=False,
        bf16=False,

        dataloader_pin_memory=False,
        report_to="none",
        remove_unused_columns=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    print("Starting training...")
    trainer.train()

    final_model_dir = os.path.join(args.output_dir, "saved_generator_model")

    trainer.save_model(final_model_dir)
    tokenizer.save_pretrained(final_model_dir)

    print("\nTraining complete.")
    print(f"Saved model to: {final_model_dir}")
    print(f"Eval split saved to: {eval_split_path}")


main_train(Args)

Loading data...
Train samples: 8500 | Eval samples: 1500
Train split saved to: /kaggle/working/t5_v11_base_generator_output/train_split.csv
Eval split saved to: /kaggle/working/t5_v11_base_generator_output/eval_split.csv
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Map:   0%|          | 0/8500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Saved model to: /kaggle/working/t5_v11_base_generator_output/saved_generator_model
Eval split saved to: /kaggle/working/t5_v11_base_generator_output/eval_split.csv


In [9]:
import json
import os
import re
import string
from collections import Counter
from dataclasses import dataclass, asdict
from difflib import SequenceMatcher
from typing import Dict, Optional

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =========================
# CONFIG
# =========================
# =========================
# CONFIG
# =========================

OUTPUT_DIR = "/kaggle/working/t5_v11_base_generator_output"

MODEL_DIR = os.path.join(
    OUTPUT_DIR,
    "saved_generator_model"
)

EVAL_SPLIT_PATH = os.path.join(
    OUTPUT_DIR,
    "eval_split.csv"
)

MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 32

DECISION_THRESHOLD = 0.72

GEN_BATCH_SIZE = 1

USE_SUBSET = False
SUBSET_SIZE = 200

# =========================
# METRICS
# =========================
def normalize_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = text.lower().strip()
    text = text.replace("\n", " ")
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str):
    return normalize_text(text).split()


def exact_match_score(pred: str, gold: str) -> float:
    return float(normalize_text(pred) == normalize_text(gold))


def token_f1_score(pred: str, gold: str) -> float:
    pred_tokens = tokenize(pred)
    gold_tokens = tokenize(gold)

    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def precision_recall_overlap(pred: str, gold: str):
    pred_tokens = tokenize(pred)
    gold_tokens = tokenize(gold)

    if not pred_tokens and not gold_tokens:
        return 1.0, 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0, 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    precision = num_same / len(pred_tokens) if pred_tokens else 0.0
    recall = num_same / len(gold_tokens) if gold_tokens else 0.0
    return precision, recall


def substring_score(pred: str, gold: str) -> float:
    p = normalize_text(pred)
    g = normalize_text(gold)

    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0

    if p in g or g in p:
        shorter = min(len(p), len(g))
        longer = max(len(p), len(g))
        return shorter / max(longer, 1)

    return 0.0


def fuzzy_ratio(pred: str, gold: str) -> float:
    try:
        from rapidfuzz import fuzz
        return float(fuzz.ratio(normalize_text(pred), normalize_text(gold))) / 100.0
    except Exception:
        return SequenceMatcher(None, normalize_text(pred), normalize_text(gold)).ratio()


def _lcs_len(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


def rouge_l_f1(pred: str, gold: str) -> float:
    try:
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
        return float(scorer.score(gold, pred)["rougeL"].fmeasure)
    except Exception:
        pred_tokens = tokenize(pred)
        gold_tokens = tokenize(gold)

        if not pred_tokens or not gold_tokens:
            return 0.0

        lcs = _lcs_len(pred_tokens, gold_tokens)
        prec = lcs / len(pred_tokens)
        rec = lcs / len(gold_tokens)
        return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)


class SemanticScorer:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.backend = None
        self.model = None
        self.vectorizer = None

        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_name)
            self.backend = "sentence_transformers"
        except Exception:
            try:
                from sklearn.feature_extraction.text import TfidfVectorizer
                self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
                self.backend = "tfidf"
            except Exception:
                self.backend = "none"

    def score(self, a: str, b: str) -> float:
        a = "" if a is None else str(a)
        b = "" if b is None else str(b)

        if self.backend == "sentence_transformers":
            emb = self.model.encode([a, b], normalize_embeddings=True)
            return float(np.dot(emb[0], emb[1]))

        if self.backend == "tfidf":
            mat = self.vectorizer.fit_transform([a, b])
            v0 = mat[0].toarray()[0]
            v1 = mat[1].toarray()[0]
            denom = np.linalg.norm(v0) * np.linalg.norm(v1)
            return float(np.dot(v0, v1) / denom) if denom > 0 else 0.0

        return 0.0


@dataclass
class AnswerComparison:
    generated_answer: str
    reference_answer: str
    exact_match: float
    token_f1: float
    overlap_precision: float
    overlap_recall: float
    substring_score: float
    fuzzy_score: float
    rouge_l_f1: float
    semantic_similarity: float
    aggregate_score: float
    quality_label: str
    is_correct_by_threshold: bool
    threshold_used: float
    similarity_to_hallucinated_answer: Optional[float] = None
    margin_vs_hallucinated: Optional[float] = None

    def to_dict(self) -> Dict:
        return asdict(self)


def compare_answers(
    generated_answer: str,
    reference_answer: str,
    hallucinated_answer: Optional[str] = None,
    semantic_scorer: Optional[SemanticScorer] = None,
    decision_threshold: float = 0.72,
) -> AnswerComparison:
    semantic_scorer = semantic_scorer or SemanticScorer()

    em = exact_match_score(generated_answer, reference_answer)
    f1 = token_f1_score(generated_answer, reference_answer)
    prec, rec = precision_recall_overlap(generated_answer, reference_answer)
    sub = substring_score(generated_answer, reference_answer)
    fuzz = fuzzy_ratio(generated_answer, reference_answer)
    rouge = rouge_l_f1(generated_answer, reference_answer)
    sem = semantic_scorer.score(generated_answer, reference_answer)

    aggregate = (
        0.20 * em
        + 0.20 * f1
        + 0.10 * prec
        + 0.10 * rec
        + 0.10 * sub
        + 0.10 * fuzz
        + 0.10 * rouge
        + 0.10 * sem
    )

    if aggregate >= 0.90:
        quality = "Excellent match"
    elif aggregate >= 0.75:
        quality = "Strong match"
    elif aggregate >= 0.55:
        quality = "Partial match"
    elif aggregate >= 0.35:
        quality = "Weak match"
    else:
        quality = "Incorrect / unsupported"

    hall_sim = None
    margin = None
    if hallucinated_answer is not None and str(hallucinated_answer).strip() != "":
        hall_sim = semantic_scorer.score(generated_answer, hallucinated_answer)
        margin = sem - hall_sim

    is_correct = aggregate >= decision_threshold

    return AnswerComparison(
        generated_answer=generated_answer,
        reference_answer=reference_answer,
        exact_match=round(em, 6),
        token_f1=round(f1, 6),
        overlap_precision=round(prec, 6),
        overlap_recall=round(rec, 6),
        substring_score=round(sub, 6),
        fuzzy_score=round(fuzz, 6),
        rouge_l_f1=round(rouge, 6),
        semantic_similarity=round(sem, 6),
        aggregate_score=round(aggregate, 6),
        quality_label=quality,
        is_correct_by_threshold=is_correct,
        threshold_used=decision_threshold,
        similarity_to_hallucinated_answer=None if hall_sim is None else round(hall_sim, 6),
        margin_vs_hallucinated=None if margin is None else round(margin, 6),
    )


# =========================
# GENERATION
# =========================
def batched_generate(model, tokenizer, texts, batch_size=2):
    preds = []
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model.to(device)
    model.eval()

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            generated_ids = model.generate(
                **encoded,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=2,
                early_stopping=True,
            )

        batch_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        preds.extend([p.strip() for p in batch_preds])

        del encoded, generated_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return preds


# =========================
# EVAL
# =========================
print("Loading eval split...")
eval_df = pd.read_csv(EVAL_SPLIT_PATH)

if USE_SUBSET:
    eval_df = eval_df.iloc[:SUBSET_SIZE].reset_index(drop=True)
    print(f"Using subset only: {len(eval_df)} samples")
else:
    print(f"Using full eval set: {len(eval_df)} samples")

print("Loading saved model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)

pred_inputs = eval_df["input_text"].tolist()

print("Generating predictions...")
preds = batched_generate(model, tokenizer, pred_inputs, batch_size=GEN_BATCH_SIZE)

print("Computing metrics...")
scorer = SemanticScorer()
results = []
correct = 0

for row, pred in zip(eval_df.to_dict("records"), preds):
    cmp = compare_answers(
        generated_answer=pred,
        reference_answer=row["right_answer"],
        hallucinated_answer=row.get("hallucinated_answer", ""),
        semantic_scorer=scorer,
        decision_threshold=DECISION_THRESHOLD,
    )

    res = {
        **row,
        "generated_answer": pred,
        **cmp.to_dict(),
    }
    results.append(res)

    if cmp.is_correct_by_threshold:
        correct += 1

result_df = pd.DataFrame(results)

pred_csv = os.path.join(OUTPUT_DIR, "eval_predictions_with_metrics.csv")
summary_json = os.path.join(OUTPUT_DIR, "eval_summary_metrics.json")

result_df.to_csv(pred_csv, index=False, encoding="utf-8-sig")

summary = {
    "num_samples": int(len(result_df)),
    "accuracy_by_threshold": float(correct / max(len(result_df), 1)),
    "exact_match_accuracy": float(result_df["exact_match"].mean()),
    "mean_token_f1": float(result_df["token_f1"].mean()),
    "mean_fuzzy_score": float(result_df["fuzzy_score"].mean()),
    "mean_semantic_similarity": float(result_df["semantic_similarity"].mean()),
    "mean_aggregate_score": float(result_df["aggregate_score"].mean()),
    "mean_margin_vs_hallucinated": float(result_df["margin_vs_hallucinated"].fillna(0).mean()),
    "threshold_used": float(DECISION_THRESHOLD),
}

with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n===== FINAL EVALUATION RESULTS =====")
for k, v in summary.items():
    print(f"{k}: {v}")

print(f"\nSaved CSV: {pred_csv}")
print(f"Saved JSON: {summary_json}")

Loading eval split...
Using full eval set: 1500 samples
Loading saved model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generating predictions...
Computing metrics...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


===== FINAL EVALUATION RESULTS =====
num_samples: 1500
accuracy_by_threshold: 0.3406666666666667
exact_match_accuracy: 0.3393333333333333
mean_token_f1: 0.4236401506666667
mean_fuzzy_score: 0.5646408793333333
mean_semantic_similarity: 0.5918500566666667
mean_aggregate_score: 0.43489058266666664
mean_margin_vs_hallucinated: 0.299565426
threshold_used: 0.72

Saved CSV: /kaggle/working/t5_v11_base_generator_output/eval_predictions_with_metrics.csv
Saved JSON: /kaggle/working/t5_v11_base_generator_output/eval_summary_metrics.json


In [10]:
import os

print(os.listdir("/kaggle/working"))

['t5_v11_base_generator_output', '.virtual_documents']
